# D1-S3-Nb6: Hit the Colour, a closed loop on the Prep

Give the Prep a colour and it finds the recipe itself. Trough 0 holds an unknown mixture of a
yellow and a blue food dye. The Prep reads it on the Byonoy, then mixes colours in batches of 16,
reads them, fits a model to every read so far, and aims the next batch at the model's best guess,
until a well matches the target.

- **Method:** computer colour matching (Allen, *J. Opt. Soc. Am.* 56:1256, 1966): the recipe comes
  from a matrix inversion of the calibrated absorbances.
- **Physics:** Beer-Lambert; the absorbances of two dyes add up.
- **Same physics, real standard:** two dyes at two wavelengths is how ISO 8655-7 (Annex B)
  verifies pipetting volumes.

**How to read this notebook:** every physical step has its own cell and runs once by hand, with
the 3D viewer open. Each cell defines the small function it runs, so the loop at the end is made
of exactly the steps you have already watched.

| Part | What happens |
|---|---|
| A | set up: modes, inputs, the Prep, the deck, the Byonoy |
| B | measure every trough with the teaching needle |
| C | the first plate: take it, lid off, target and blank in, read the target |
| D | batch 1, step by step: yellow, blue, water, mix, read, model |
| E | the loop: the same steps, batch after batch, a new plate when one is full |
| F | the result, and shutting down |

---
# A. Set up
## Modes
The Prep and the reader each run simulated or real. A simulated reader computes each well's
absorbance from what the Prep pipetted into it; a seed makes a simulated run repeatable.

In [2]:
prep_simulation = False  # True or False
reader_simulation = False  # True or False
seed = 42  # simulation: draws the hidden target and all scatter; the same seed, the same run

run_identifier = "nb6"

## Master inputs

In [3]:
well_uL = 100  # every well: the target, or yellow + blue + water
wavelengths = [450, 600]  # yellow absorbs at 450 nm, blue at 600 nm
dye_range_uL = (0.0, 20.0)  # how much of each dye a recipe may take
batches_per_plate = 5  # 16 colours each, in columns 3 to 12
max_plates = 2  # a full plate goes to the trash and the next comes off the stack
plates_in_stack = 4  # lidded plates on the stack at the start
tolerance = 0.01  # stop when a well is this close to the target (absorbance at both wavelengths)
above_surface_mm = 3.0  # dispense this far above the liquid: a tip never touches another liquid

## Logging

In [4]:
import datetime
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

prep_mode = "simulation" if prep_simulation else "execution"
reader_mode = "simulation" if reader_simulation else "execution"
protocol_mode = f"prep_{prep_mode}-reader_{reader_mode}"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)
pylabrobot.verbose(True, level=logging.INFO)
print(f"logging to {log_path}")

logging to _logs/prep_execution-reader_execution/nb6_20260928-152619.log


## The 3D viewer

In [5]:
from pylabrobot.resources import Coordinate
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=1_000, size_y=1_000, size_z=0)
viewer = Viewer3D(root=facility, name="nb6_hit_the_colour")
await viewer.start()

viewer on http://127.0.0.1:1341/#token=pzrzmeVmqsEZP3Ve7ZZodsqAGjXAjTys93a-pqfkDqU  (websocket 2125)


viewer: a browser connected, drawing with WebGPU


## The Prep
2 independent channels, an 8-channel head, CoRe grippers.

In [ ]:
from pylabrobot.hamilton.prep import Prep

prep = Prep(simulation=prep_simulation, host=None if prep_simulation else "192.168.100.102")
facility.assign_child_resource(prep, location=Coordinate(x=0.0, y=0.0, z=0.0))

await prep.setup()
await prep.lights.turn_on()

ValueError: Resource with name 'Prep' already exists in the tree.

In [51]:
await prep.setup()


2026-09-28 15:45:51,163 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 15:45:51,590 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 15:45:51,592 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton root objects...
2026-09-28 15:45:51,594 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton global objects...
2026-09-28 15:45:51,598 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client setup complete. Client ID: 17, globals: 1
2026-09-28 15:45:51,662 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 15:45:52,162 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 15:45:52,165 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton root objects...
2026-09-28 15:45:52,167 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton global object

2026-09-28 15:55:52,635 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:17:65535 seq=0 action=0x7
2026-09-28 15:55:52,636 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:18:65535 seq=0 action=0x7
2026-09-28 15:55:52,637 - pylabrobot.hamilton.transport.tcp.session - WARNING - Hamilton TCP reader stopped: Connection closed before num_bytes are read
2026-09-28 15:56:02,137 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:17:65535 seq=0 action=0x7
2026-09-28 15:56:02,138 - pylabrobot.hamilton.transport.tcp.session - WARNING - Hamilton TCP reader stopped: Connection closed before num_bytes are read


## The deck

| Site | Holds |
|---|---|
| 0 | Byonoy detection unit: the plate in use sits on it for the whole plate |
| 1 | 50 uL tips (2 channels): a pair for each liquid, each batch; a rack per run |
| 2 | Byonoy parking unit: the illumination unit waits here while the Prep pipettes |
| 4 (`slot_1_0`) | trough rack: target (0), water (1), yellow (2), blue (3) |
| 5 | 300 uL tips (8-channel head) for the second plate |
| 6 | 300 uL tips (8-channel head) for the first plate |
| 7 | a stack of lidded plates (Corning 96, flat bottom) |

In [7]:
import numpy as np
import pandas as pd

from pylabrobot.resources import (
    Cor_96_wellplate_360ul_Fb,
    ResourceStack,
    set_tip_tracking,
    set_volume_tracking,
)
from pylabrobot.resources.hamilton import (
    hamilton_1_trough_60mL_Vb,
    hamilton_96_tiprack_50uL_NTR,
    hamilton_96_tiprack_300uL_NTR,
)

from _helpers.sdl_helper import hamilton_5_troughrack_60mL

set_tip_tracking(enabled=True)
set_volume_tracking(enabled=True)

tips_50uL = prep.deck[1] = hamilton_96_tiprack_50uL_NTR(name="tips_50uL")
tips_300uL_plate_1 = prep.deck[6] = hamilton_96_tiprack_300uL_NTR(name="tips_300uL_plate_1")
tips_300uL_plate_2 = prep.deck[5] = hamilton_96_tiprack_300uL_NTR(name="tips_300uL_plate_2")

plate_stack = prep.deck[7] = ResourceStack(name="plate_stack", direction="z")
for idx in range(plates_in_stack):
    plate_stack.assign_child_resource(Cor_96_wellplate_360ul_Fb(name=f"colour_plate_{idx}", with_lid=True))

troughs = prep.deck[4] = hamilton_5_troughrack_60mL(name="troughs")
troughs[0] = target_trough = hamilton_1_trough_60mL_Vb(name="target")
troughs[1] = water = hamilton_1_trough_60mL_Vb(name="water")
troughs[2] = yellow = hamilton_1_trough_60mL_Vb(name="yellow")
troughs[3] = blue = hamilton_1_trough_60mL_Vb(name="blue")
for trough in (target_trough, water, yellow, blue):
    trough.tracker.set_volume(20_000)  # measured in part B when running for real

/var/folders/_3/0_4lk1cx741663czqtjw50qh0000gq/T/ipykernel_52608/1330740969.py:27: DeprecationWarning: Cor_96_wellplate_360ul_Fb() is deprecated and will be removed in v1b1. Use cor_96_wellplate_360uL_Fb() instead.
  plate_stack.assign_child_resource(Cor_96_wellplate_360ul_Fb(name=f"colour_plate_{idx}", with_lid=True))


## The Byonoy on the deck
The detection unit reads, and holds the plate. The illumination unit sits on top of it to read,
and on the parking unit while the Prep pipettes. A plate's lid rests on top of the illumination
unit, and travels with it.

In [8]:
from pylabrobot.byonoy import (
    byonoy_a96a_detection_unit,
    byonoy_a96a_illumination_unit,
    byonoy_a96a_parking_unit,
    byonoy_sbs_adapter,
)

adapter_0 = prep.deck[0] = byonoy_sbs_adapter(name="byonoy_sbs_adapter_0")
adapter_2 = prep.deck[2] = byonoy_sbs_adapter(name="byonoy_sbs_adapter_2")

reader = byonoy_a96a_detection_unit(name="a96a_detection_unit")
parking = byonoy_a96a_parking_unit(name="a96a_parking_unit")
on_adapter = Coordinate(x=-23.5, y=-(95.48 - 85.48) / 2, z=17.0)
adapter_0.assign_child_resource(reader, location=on_adapter)
adapter_2.assign_child_resource(parking, location=on_adapter)

illumination = byonoy_a96a_illumination_unit(name="a96a_illumination_unit")
reader.illumination_unit_holder.assign_child_resource(illumination)

2026-09-28 15:26:31,680 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [9]:
from _helpers.sdl_helper import SimulatedReader

if reader_simulation:  # the hidden target: a seeded random recipe, per 100 uL
    rng = np.random.default_rng(seed)
    simulated_target_uL = dict(zip(["yellow", "blue"], rng.uniform(2.0, 18.0, size=2)))
    simulated_reader = SimulatedReader(
        troughs={  # uL of dye stock per uL of each trough's liquid
            "target": {dye: uL / well_uL for dye, uL in simulated_target_uL.items()},
            "water": {},
            "yellow": {"yellow": 1.0},
            "blue": {"blue": 1.0},
        },
        seed=seed,
    )
else:
    await reader.setup(ignore_errors_in_calibration=True)
    missing = [wl for wl in wavelengths if wl not in reader.available_wavelengths]
    assert not missing, f"the reader has no {missing} nm filter: {reader.available_wavelengths}"

2026-09-28 15:26:31,693 - pylabrobot.byonoy.driver - INFO - [a96a_detection_unit] connected
2026-09-28 15:26:31,702 - pylabrobot.byonoy.absorbance_96 - WARNING - [a96a_detection_unit] calibration-error override enabled; calibration may be invalid
2026-09-28 15:26:36,031 - pylabrobot.byonoy.driver - WARNING - [a96a_detection_unit] non-zero chunk flags during read: ['0x10'] (vendor bit definitions not published; data may be unreliable)
2026-09-28 15:26:36,039 - pylabrobot.byonoy.absorbance_96 - INFO - [a96a_detection_unit] ready, available wavelengths: [600, 450] nm


[600, 450]

In [10]:
await prep.core_grippers.pick_up_tools()

In [27]:
a96a_illumination_unit = prep.get_resource("a96a_illumination_unit")
a96a_parking_unit = prep.get_resource("a96a_parking_unit")
a96a_detection_unit = prep.get_resource("a96a_detection_unit")

_ByonoyAbsorbanceIlluminationUnitHolder(name='a96a_parking_unit_illumination_unit_holder', location=Coordinate(000.000, 000.000, 014.100), size_x=155.26, size_y=95.48, size_z=0, category=resource_holder)

In [14]:
await prep.core_grippers.pick_up_resource(
    resource=a96a_illumination_unit,
    pickup_distance_from_top=15.0
)

In [20]:
await prep.core_grippers.drop_resource(
    to=a96a_parking_unit.illumination_unit_holder,
)

2026-09-28 15:30:02,707 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [23]:
prep.get_resource("colour_plate_3").get_location_wrt(prep.deck)

Coordinate(x=138.67, y=286.49, z=50.0)

In [24]:
await prep.core_grippers.pick_up_resource(
    resource=prep.get_resource("colour_plate_3"),
    pickup_distance_from_top=12.0
)

In [28]:
await prep.core_grippers.drop_resource(
    to=a96a_detection_unit.plate_holder,
)

In [29]:
await prep.core_grippers.pick_up_resource(
    resource=prep.get_resource("colour_plate_3_lid"),
    pickup_distance_from_top=3
)

In [30]:
await prep.core_grippers.drop_resource(
    to=a96a_illumination_unit,
    offset=Coordinate(5,5,a96a_illumination_unit.get_size_z())
)

In [31]:
await prep.core_grippers.pick_up_resource(
    resource=a96a_illumination_unit,
    pickup_distance_from_top=15
)

In [33]:
await prep.core_grippers.drop_resource(
    to=a96a_detection_unit.illumination_unit_holder,
)

2026-09-28 15:34:59,738 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [37]:
prep.get_resource("colour_plate_3")["A4:G4"]

[Well(name='colour_plate_3_well_A4', location=Coordinate(037.870, 070.770, 003.030), size_x=6.86, size_y=6.86, size_z=10.67, category=well),
 Well(name='colour_plate_3_well_B4', location=Coordinate(037.870, 061.770, 003.030), size_x=6.86, size_y=6.86, size_z=10.67, category=well),
 Well(name='colour_plate_3_well_C4', location=Coordinate(037.870, 052.770, 003.030), size_x=6.86, size_y=6.86, size_z=10.67, category=well),
 Well(name='colour_plate_3_well_D4', location=Coordinate(037.870, 043.770, 003.030), size_x=6.86, size_y=6.86, size_z=10.67, category=well),
 Well(name='colour_plate_3_well_E4', location=Coordinate(037.870, 034.770, 003.030), size_x=6.86, size_y=6.86, size_z=10.67, category=well),
 Well(name='colour_plate_3_well_F4', location=Coordinate(037.870, 025.770, 003.030), size_x=6.86, size_y=6.86, size_z=10.67, category=well),
 Well(name='colour_plate_3_well_G4', location=Coordinate(037.870, 016.770, 003.030), size_x=6.86, size_y=6.86, size_z=10.67, category=well)]

In [46]:
import time

In [48]:
start_time = time.time()

data = await reader.read_absorbance(
    plate=prep.get_resource("colour_plate_3"),
    wells=prep.get_resource("colour_plate_3").get_all_children(),
    wavelength=450
)

time.time() - start_time

2026-09-28 15:42:52,221 - pylabrobot.byonoy.absorbance_96 - INFO - [a96a_detection_unit] reading absorbance: plate='colour_plate_3', wavelength=450 nm, wells=96/96
2026-09-28 15:42:53,618 - pylabrobot.byonoy.driver - WARNING - [a96a_detection_unit] non-zero chunk flags during read: ['0x01', '0x01', '0x01', '0x01', '0x01', '0x01', '0x01', '0x01'] (vendor bit definitions not published; data may be unreliable)


1.398163080215454

In [ ]:
pd.DataFrame((data[0].data)).to_

,0,1,2,3,4,5,6,7,8,9,10,11
0,0.037286,0.037446,0.038182,0.038290,0.038526,0.037046,0.038112,0.039170,0.039848,0.038959,0.040583,0.037794
1,0.038037,0.037818,0.039087,0.038841,0.038768,0.038691,0.038459,0.040236,0.039466,0.041161,0.039480,0.037595
2,0.038034,0.038743,0.039045,0.040150,0.040085,0.038889,0.039389,0.039068,0.040724,0.040396,0.041888,0.040279
3,0.040260,0.039371,0.040102,0.040362,0.040358,0.037229,0.037536,0.039301,0.040150,0.040276,0.039301,0.037374
4,0.039572,0.040225,0.040079,0.041027,0.039436,0.037634,0.037228,0.039720,0.039451,0.039087,0.039298,0.037050
5,0.044495,0.044570,0.040183,0.040935,0.040008,0.038258,0.038919,0.039644,0.040187,0.039180,0.037912,0.036558
6,0.040480,0.038775,0.046070,0.038807,0.039667,0.038193,0.037999,0.039025,0.038463,0.038655,0.037184,0.036593
7,0.038546,0.038141,0.048021,0.037763,0.038468,0.037060,0.036023,0.038328,0.036383,0.037023,0.036126,0.036623


---
# B. Measure every trough
The teaching needle is metal, so capacitive detection finds the liquid in each trough: 3 probes
each, and the volume found is what the model tracks from here. Every later aspirate goes to the
height that volume gives and follows the surface down, with no liquid-level detection at all: the
tips can be transparent. In simulation there is no liquid to find, so the volumes stay as set.

In [ ]:
LLD = prep.pipettes.LLDMode
needle = prep.get_resource("Prep_teaching_needle")
all_troughs = [target_trough, water, yellow, blue]

if not prep_simulation:
    await prep.pipettes.pick_up_tips(tip_spots=[needle], use_channels=[0])
    for trough in all_troughs:
        volumes = await prep.pipettes.probe_liquid_volumes(
            containers=[trough], use_channels=[0], lld_mode=LLD.CAPACITIVE, n_replicates=3
        )
        trough.tracker.set_volume(volumes[0])
    await prep.pipettes.drop_tips(destinations=[needle], use_channels=[0])

pd.DataFrame({"uL": [t.tracker.get_used_volume() for t in all_troughs]}, index=[t.name for t in all_troughs])

In [ ]:
def liquid_height(container):
    """Where the liquid stands in a trough or well, in mm above its bottom, from its tracked volume."""
    return container.compute_height_from_volume(container.tracker.get_used_volume())

---
# C. The first plate, step by step
## C1. The illumination unit onto the parking unit
The grippers take their tools, move the illumination unit off the detection unit, and put the
tools back.

In [ ]:
async def move_illumination_unit(to):
    await prep.core_grippers.pick_up_tools()
    await prep.core_grippers.move_resource(
        resource=illumination, to=to, pickup_distance_from_top=15.4,
        pickup_y_clearance=2.0, drop_y_clearance=2.0, drop_z_acceleration=50, return_grippers=False,
    )
    await prep.core_grippers.return_tools()


await move_illumination_unit(parking.illumination_unit_holder)

## C2. A plate off the stack onto the detection unit

In [ ]:
async def take_plate():
    plate = plate_stack.get_top_item()
    await prep.core_grippers.pick_up_tools()
    await prep.core_grippers.move_resource(
        resource=plate, to=reader.plate_holder, pickup_distance_from_top=12.0,
        drop_z_acceleration=50, return_grippers=False,
    )
    await prep.core_grippers.return_tools()
    if reader_simulation:
        simulated_reader.new_plate()
    return plate


plate = await take_plate()

## C3. Its lid on top of the illumination unit

In [ ]:
async def lid_onto_illumination_unit():
    lid = plate.lid
    await prep.core_grippers.pick_up_tools()
    await prep.core_grippers.move_resource(
        resource=lid, to=illumination, pickup_distance_from_top=4.0,
        drop_z_acceleration=50, return_grippers=False,
    )
    await prep.core_grippers.return_tools()
    return lid


lid = await lid_onto_illumination_unit()

## C4. The target into column 1
The 8-channel head takes the next full column of 300 uL tips the rack has (`find_tip_spots`):
100 uL of the unknown mixture from trough 0 into all of column 1, then the tips go to the waste.

In [ ]:
from pylabrobot.lib.liquid_handling.tip_spot_finding import find_tip_spots

ROWS = "ABCDEFGH"
tips_300uL = tips_300uL_plate_1  # this plate's rack of 300 uL tips
target_wells = [f"{row}1" for row in ROWS]
blank_wells = [f"{row}2" for row in ROWS]


def head_tips():
    """The next full column of tips in this plate's rack, for the 8-channel head."""
    return find_tip_spots(tips_300uL, has_tip=True, count=8, x_aligned=True)


async def fill_column(trough, column):
    await prep.head8.pick_up_tips(head_tips())
    await prep.head8.aspirate(
        [trough], volume=well_uL, liquid_height=liquid_height(trough),
        lld_mode=LLD.OFF, auto_surface_following=True,
    )
    await prep.head8.dispense(plate[f"A{column}:H{column}"], volume=well_uL)
    await prep.head8.discard_tips()
    if reader_simulation:
        for row in ROWS:
            simulated_reader.add(f"{row}{column}", trough.name, well_uL)


await fill_column(trough=target_trough, column=1)

## C5. The blank into column 2
The same, with water: the reads are measured against it.

In [ ]:
await fill_column(trough=water, column=2)

## C6. Read the target against the blank
The illumination unit goes on, the reader reads columns 1 and 2 at both wavelengths, and the
illumination unit goes back to the parking unit. The target is the mean of column 1 minus the mean
of column 2.

In [ ]:
async def read(well_names):
    """Raw absorbance of the wells at each wavelength; the illumination unit is on."""
    raw = {}
    for wl in wavelengths:
        if reader_simulation:
            raw[wl] = simulated_reader.read(well_names, wl)
        else:
            matrix = (await reader.read_absorbance(plate, wells=plate.get_all_items(), wavelength=wl))[0].data
            raw[wl] = {n: matrix[ROWS.index(n[0])][int(n[1:]) - 1] for n in well_names}
    return pd.DataFrame({f"A{wl}": raw[wl] for wl in wavelengths})


await move_illumination_unit(reader.illumination_unit_holder)
raw = await read(target_wells + blank_wells)
await move_illumination_unit(parking.illumination_unit_holder)

target_reads = raw.loc[target_wells] - raw.loc[blank_wells].mean()
target = target_reads.mean().to_numpy()
target_reads.agg(["mean", "std"]).round(4)

## C7. Open the live window
A tkinter window that redraws after every batch: each batch's 16 colours beside the target, the
search closing in on the recipe, and the best distance per batch.

In [ ]:
from _helpers.sdl_helper import open_progress_window

show_progress, close_progress = open_progress_window(target, tolerance)
batches = []  # every batch so far: its recipes, reads and distances

---
# D. Batch 1, step by step
## D1. The 16 recipes
A 4 x 4 grid over the whole space, 0 to 20 uL of each dye, into columns 3 and 4. These reads also
calibrate the model.

In [ ]:
from _helpers.sdl_helper import grid_batch

low, high = dye_range_uL
width = high - low
recipes = grid_batch(((low + high) / 2, (low + high) / 2), width, low, high)
columns = [3, 4]
wells = [f"{row}{c}" for c in columns for row in ROWS]

pd.DataFrame(recipes, index=wells, columns=["uL yellow", "uL blue"]).round(1).T

## D2. Yellow
Both channels, two wells at a time, dispensed from above: the tips never touch the liquid in a well,
so one pair of tips does the whole batch.

In [ ]:
import math

channels = [0, 1]


def passes(uL):
    """A dose in passes of at most 50 uL, the tips' volume: 80 uL -> [40, 40], 15 uL -> [15]."""
    n = math.ceil(uL / 50)
    return [uL / n] * n


async def add_liquid(trough, doses):
    """doses: [(well name, uL)]. Both channels, two wells at a time, dispensed from above the liquid,
    so one pair of tips does it all: a tip never touches the liquid in a well."""
    split = [(name, passes(uL)) for name, uL in doses if uL > 0]
    rounds = [
        [(name, p[i]) for name, p in split if i < len(p)] for i in range(max(len(p) for _, p in split))
    ]  # every well's first pass, then every well's second pass
    tips = find_tip_spots(tips_50uL, has_tip=True, count=2, x_aligned=True)
    await prep.pipettes.pick_up_tips(tip_spots=tips, use_channels=channels)
    for moves in rounds:
        for i in range(0, len(moves), 2):
            pair = moves[i : i + 2]
            use = channels[: len(pair)]
            wells = [plate[name][0] for name, _ in pair]
            volumes = [uL for _, uL in pair]
            await prep.pipettes.aspirate(
                containers=[trough] * len(use), volumes=volumes, use_channels=use,
                liquid_heights=[liquid_height(trough)] * len(use), lld_mode=LLD.OFF,
                auto_surface_following=True, blow_out=[True] * len(use),
            )
            await prep.pipettes.dispense(
                containers=wells, volumes=volumes, use_channels=use,
                liquid_heights=[liquid_height(w) for w in wells],
                immersion_depths=[-above_surface_mm] * len(use), blow_out=[True] * len(use),
            )
            if reader_simulation:
                for name, uL in pair:
                    simulated_reader.add(name, trough.name, uL)
    await prep.pipettes.discard_tips(use_channels=channels)


await add_liquid(yellow, [(n, y) for n, (y, b) in zip(wells, recipes)])

## D3. Blue

In [ ]:
await add_liquid(blue, [(n, b) for n, (y, b) in zip(wells, recipes)])

## D4. Water, to 100 uL in every well

In [ ]:
await add_liquid(water, [(n, well_uL - y - b) for n, (y, b) in zip(wells, recipes)])

## D5. Mix
The 8-channel head mixes a column at a time, with the next full column of tips for each: the
dyes and the water are in, so every well is mixed before it is read.

In [ ]:
from pylabrobot.lib.liquid_handling.mix import Mix

async def mix_column(column):
    await prep.head8.pick_up_tips(head_tips())
    await prep.head8.mix(plate[f"A{column}:H{column}"], Mix(volume=50, repetitions=5, flow_rate=100))
    await prep.head8.discard_tips()


for column in columns:
    await mix_column(column)

## D6. Read
The illumination unit on, read the batch and the blank column, the illumination unit off. Every
read is blank-subtracted.

In [ ]:
async def read_batch(wells):
    await move_illumination_unit(reader.illumination_unit_holder)
    raw = await read(wells + blank_wells)
    await move_illumination_unit(parking.illumination_unit_holder)
    return raw.loc[wells] - raw.loc[blank_wells].mean()


reads = await read_batch(wells)
reads.round(3).T

## D7. How close did it get?
Each well's distance to the target, at both wavelengths together. The live window updates.

In [ ]:
def record(recipes, wells, reads, plate_number):
    """Keep the batch, redraw the live window, and say how close its best well came."""
    distances = np.linalg.norm(reads.to_numpy() - target, axis=1)
    batches.append({"plate": plate_number, "wells": wells, "recipes": recipes,
                    "reads": reads.to_numpy(), "distances": distances})
    best = int(np.argmin(distances))
    show_progress(batches)
    print(f"batch {len(batches)} (plate {plate_number}): best {wells[best]}, "
          f"{recipes[best][0]:.1f} uL yellow, {recipes[best][1]:.1f} uL blue, "
          f"distance {distances[best]:.4f}")
    return distances[best]


best_distance = record(recipes, wells, reads, plate_number=1)

## D8. The model, and the next 16 recipes
The model: absorbance per uL of each dye at each wavelength, fitted on every read so far (a least-
squares line through zero, Beer-Lambert). Solving it for the target gives the best guess; the next
16 recipes sit in a grid around it, half as wide.

In [ ]:
from _helpers.sdl_helper import fit_model, solve_recipe


def next_recipes(width):
    """Fit the model on every batch so far, solve it for the target, and grid the next 16."""
    model = fit_model(
        np.vstack([b["recipes"] for b in batches]), np.vstack([b["reads"] for b in batches])
    )
    guess = np.clip(solve_recipe(model, target), low, high)
    print(f"model's guess: {guess[0]:.1f} uL yellow, {guess[1]:.1f} uL blue")
    return grid_batch(guess, width, low, high)


width /= 2
recipes = next_recipes(width)

---
# E. The loop
### A batch
D1 to D8, the same calls as above, on the next two columns.

In [ ]:
async def run_batch(columns, plate_number):
    global recipes, width
    wells = [f"{row}{c}" for c in columns for row in ROWS]  # D1
    await add_liquid(yellow, [(n, y) for n, (y, b) in zip(wells, recipes)])  # D2
    await add_liquid(blue, [(n, b) for n, (y, b) in zip(wells, recipes)])  # D3
    await add_liquid(water, [(n, well_uL - y - b) for n, (y, b) in zip(wells, recipes)])  # D4
    for column in columns:  # D5
        await mix_column(column)
    reads = await read_batch(wells)  # D6
    distance = record(recipes, wells, reads, plate_number)  # D7
    width /= 2  # D8
    recipes = next_recipes(width)
    return distance


async def run_plate(first_column, plate_number):
    """Batches on this plate, two columns each, until a well matches or the plate is full."""
    for left in range(first_column, 13, 2):
        if await run_batch([left, left + 1], plate_number) < tolerance:
            return True
    return False

### When a plate is full
The lid goes back on the plate, the grippers throw the closed plate into the trash past the waste
block (`discard_resource`), and the next plate starts the way the first did: C2 to C6, with its
own rack of 300 uL tips. The model keeps every batch.

In [ ]:
async def change_plate():
    global plate, lid, tips_300uL, target
    await prep.core_grippers.pick_up_tools()
    await prep.core_grippers.move_resource(  # the lid back on: the plate goes out closed
        resource=lid, to=plate, pickup_distance_from_top=4.0,
        drop_z_acceleration=50, return_grippers=False,
    )
    await prep.core_grippers.discard_resource(plate, pickup_distance_from_top=12.0)
    await prep.core_grippers.return_tools()

    plate = await take_plate()  # C2
    lid = await lid_onto_illumination_unit()  # C3
    tips_300uL = tips_300uL_plate_2
    await fill_column(trough=target_trough, column=1)  # C4
    await fill_column(trough=water, column=2)  # C5
    await move_illumination_unit(reader.illumination_unit_holder)  # C6
    raw = await read(target_wells + blank_wells)
    await move_illumination_unit(parking.illumination_unit_holder)
    target = (raw.loc[target_wells] - raw.loc[blank_wells].mean()).mean().to_numpy()

### Run it
The rest of the first plate (batch 1 used columns 3 and 4), then a new plate while none matches.

In [ ]:
matched = best_distance < tolerance or await run_plate(first_column=5, plate_number=1)
for plate_number in range(2, max_plates + 1):
    if matched:
        break
    await change_plate()
    matched = await run_plate(first_column=3, plate_number=plate_number)

print("matched" if matched else "not matched: the plates are full")

---
# F. The result

In [ ]:
import os

import matplotlib.pyplot as plt

from _helpers.sdl_helper import draw_progress

if reader_simulation:
    print("hidden target:", {d: round(float(v), 2) for d, v in simulated_target_uL.items()})

fig = plt.figure(figsize=(13, 5))
draw_progress(fig, target, tolerance, batches)
plt.show()

rows = [
    {"batch": i + 1, "plate": b["plate"], "well": w, "uL yellow": y, "uL blue": bl,
     "A450": r[0], "A600": r[1], "distance": d}
    for i, b in enumerate(batches)
    for w, (y, bl), r, d in zip(b["wells"], b["recipes"], b["reads"], b["distances"])
]
results = pd.DataFrame(rows)
os.makedirs("_results", exist_ok=True)
results_path = f"_results/{run_identifier}_{started_at}_hit_the_colour.csv"
results.to_csv(results_path, index=False)
print(f"results written to {results_path}")
results.loc[results.groupby("batch")["distance"].idxmin()].set_index("batch").round(4)

## Close the live window

In [ ]:
close_progress()

---
# Shutdown

In [49]:
await prep.stop()

if not reader_simulation:
    await reader.stop()

await viewer.stop()

2026-09-28 15:45:06,904 - pylabrobot.hamilton.prep.driver.master - ERROR - could not read whether a plate is held; treating it as held
Traceback (most recent call last):
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/driver/master.py", line 1215, in _return_held_resource
    if not await self._request_plate_held():
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/driver/master.py", line 1245, in _request_plate_held
    return bool((await self.send_command(PrepCmd.PrepGetPlateHeld())).value)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/driver/master.py", line 789, in send_command
    resolved = await self._resolve_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/dr